# ShortForge AI — Kaggle Production GPU Worker (Minecraft & Geography)

This notebook provides a complete cloud rendering environment for ShortForge AI:
- **Minecraft Story Studio**: Original 3D animated voxel films (e.g. 'The Villager Who Had Only One Heart') with Blender keyframes, articulated characters, Kokoro voiceover, SFX, and subtitles.
- **Geography Studio**: GeoGlobeTales-style 3D rotating Earth explainers with Natural Earth datasets.

### Instructions:
1. **Accelerator**: Select GPU (T4 x2 or P100) in Kaggle sidebar settings.
2. **Internet**: Toggle Internet switch **ON** in Kaggle sidebar settings.
3. **Run All**: Execute cells sequentially to verify Blender, pre-fetch assets, run a fast preview, and render the final vertical short.
4. **Download**: All generated videos are packaged at `/kaggle/working/shortforge_outputs.zip` for instant download.

In [ ]:
# =============================================================================
# 1. CONFIGURATION
# =============================================================================
REPO_URL = "https://github.com/Mayyank04/shortsforge.git"
ZIP_PATH = ""  # e.g. "/kaggle/input/shortforge-ai/shortforge-ai.zip" (if using uploaded zip)

# Mode: "minecraft" (Minecraft Story Studio) or "geo" (Geography Studio)
MODE = "minecraft"
TOPIC = "The Villager Who Had Only One Heart"
GENRE = "emotional"          # emotional, adventure, friendship, survival, mystery, horror, etc.
STYLE = "geo_realistic"      # Used if MODE == "geo"
VOICE = "heart"              # heart, adam, alice, george, nicole, michael
LANGUAGE = "en"              # en (English), hi (Hindi)
DURATION = 50                # 50s for Minecraft (10 scenes), 75s for Geography
RUN_PREVIEW_FIRST = True     # Fast preview before full 1080x1920

API_URL = ""                 # Optional deployed FastAPI URL
JOB_ID = ""                  # Optional queued dashboard job ID
DAEMON_MODE = False          # Continuous background worker mode


In [ ]:
# =============================================================================
# 2. REPOSITORY SETUP (GIT CLONE OR ZIP EXTRACTION)
# =============================================================================
import os, sys, shutil, zipfile
from pathlib import Path

WORKDIR = Path("/kaggle/working/shortforge-ai")

if WORKDIR.exists():
    print(f"Working directory {WORKDIR} already exists.")
elif ZIP_PATH and Path(ZIP_PATH).exists():
    print(f"Extracting repository from ZIP: {ZIP_PATH} ...")
    WORKDIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(WORKDIR)
    print("Extracted successfully!")
elif REPO_URL:
    print(f"Cloning repository from {REPO_URL} ...")
    try:
        ret = os.system(f"git clone {REPO_URL} {WORKDIR}")
        if ret != 0:
            raise RuntimeError(f"Git clone failed with exit code {ret}.")
    except Exception as exc:
        print(f"Error cloning repository: {exc}")
        print("\nNOTE: If your repository is private, either:")
        print("  1. Upload the project folder as a ZIP in Kaggle Datasets and set ZIP_PATH = '/kaggle/input/...'")
        print("  2. Or embed a Personal Access Token in REPO_URL: 'https://<token>@github.com/username/repo.git'")
        raise
else:
    raise ValueError("Please provide either a valid REPO_URL or ZIP_PATH in the configuration cell.")

os.chdir(WORKDIR)
print(f"Active working directory: {os.getcwd()}")


In [ ]:
# =============================================================================
# 3. SYSTEM PACKAGES INSTALLATION
# =============================================================================
print("Installing system packages (FFmpeg, Blender, espeak-ng, OpenGL)...\n")
!apt-get update -qq && apt-get install -y --no-install-recommends ffmpeg blender espeak-ng libgl1-mesa-glx > /dev/null

print("Verifying system binaries:")
!ffmpeg -version | head -n 1
!blender --version | head -n 1


In [ ]:
# =============================================================================
# 4. PYTHON DEPENDENCIES INSTALLATION
# =============================================================================
print("Installing ShortForge AI requirements...\n")
!pip install -q --upgrade pip
!pip install -q -r backend/requirements.txt -r worker/requirements.txt


In [ ]:
# =============================================================================
# 5. GEOGRAPHICAL ASSETS & BLENDER VERIFICATION
# =============================================================================
import os
from pathlib import Path
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"
Path("/kaggle/working/outputs").mkdir(parents=True, exist_ok=True)

from worker.blender_render import verify_blender_available
from worker.geo.data_manager import ensure_geo_assets, generate_default_audio_assets

bin_path, version = verify_blender_available()
print(f"✓ Blender verified: {bin_path} ({version})")

print("Checking and caching geographical datasets and textures...")
assets = ensure_geo_assets()
print(f"✓ All {len(assets)} geographical datasets and textures verified.")

generate_default_audio_assets()
print("✓ Default audio effects and ambient soundtrack verified.")


In [ ]:
# =============================================================================
# 6. VOICE SYNTHESIS CONFIGURATION
# =============================================================================
import os
from pathlib import Path

use_kokoro = False
try:
    import kokoro, soundfile
    print("Kokoro TTS and soundfile available. Running voice probe...")
    from worker.geo_voice import synthesize_narration
    probe_wav = Path("/tmp/shortforge_probe.wav")
    synthesize_narration("ShortForge geography engine active.", probe_wav, voice_name=VOICE)
    if probe_wav.exists() and probe_wav.stat().st_size > 1000:
        use_kokoro = True
        print("✓ Kokoro neural TTS initialized successfully.")
except Exception as exc:
    print(f"Notice: Kokoro probe encountered: {exc}")

if not use_kokoro:
    print("Using espeak-ng system TTS fallback (TTS_ENGINE=system).")
    os.environ["TTS_ENGINE"] = "system"
else:
    os.environ.pop("TTS_ENGINE", None)


In [ ]:
# =============================================================================
# 7. FAST PREVIEW RENDER (540x960)
# =============================================================================
if RUN_PREVIEW_FIRST and not JOB_ID and not DAEMON_MODE:
    print(f"=== Running Preview Render ({MODE.upper()} mode, Topic: '{TOPIC}') ===")
    !python -m worker.cli --mode "$MODE" --topic "$TOPIC" --genre "$GENRE" --style "$STYLE" --duration $DURATION --voice "$VOICE" --language "$LANGUAGE" --preview --output-dir /kaggle/working/outputs
    print("✓ Preview render finished successfully.")


In [ ]:
# =============================================================================
# 8. PRODUCTION RENDER (1080x1920)
# =============================================================================
print(f"=== Running Production Render ===")
if DAEMON_MODE:
    print(f"Starting background worker daemon polling {API_URL}...")
    !python -m worker.cli --daemon --api-url "$API_URL" --output-dir /kaggle/working/outputs
elif JOB_ID and API_URL:
    print(f"Rendering queued API job {JOB_ID}...")
    !python -m worker.cli --api-url "$API_URL" --job-id "$JOB_ID" --output-dir /kaggle/working/outputs
else:
    print(f"Rendering topic: '{TOPIC}' (mode={MODE}, genre={GENRE}, duration={DURATION}s, voice={VOICE})...")
    !python -m worker.cli --mode "$MODE" --topic "$TOPIC" --genre "$GENRE" --style "$STYLE" --duration $DURATION --voice "$VOICE" --language "$LANGUAGE" --output-dir /kaggle/working/outputs


In [ ]:
# =============================================================================
# 9. OUTPUT DISPLAY & DOWNLOAD ARCHIVE
# =============================================================================
from IPython.display import Video, FileLink, display
import glob, shutil
from pathlib import Path

out_dir = Path("/kaggle/working/outputs")
mp4s = sorted(out_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)

if mp4s:
    latest = mp4s[-1]
    mb = latest.stat().st_size / (1024 * 1024)
    print(f"✓ Generated Video: {latest.name} ({mb:.2f} MB)")
    display(Video(str(latest), embed=True, width=360))
    
    # Create ZIP archive for download
    archive_path = Path("/kaggle/working/shortforge_outputs.zip")
    shutil.make_archive("/kaggle/working/shortforge_outputs", "zip", str(out_dir))
    print(f"\nDownload ready: {archive_path} ({archive_path.stat().st_size / (1024*1024):.2f} MB)")
    display(FileLink(str(archive_path)))
else:
    print("No MP4 files found in /kaggle/working/outputs.")
